# SQANTI filter worksheet

This worksheet is designed to guide you through the analysis of SQANTI3 Quality Control outputs, focusing on both the basic and complete classification.txt files. In the first part, you will examine each classification independently to understand how and why isoforms are assigned specific categories. In the second part, you will compare the two classifications to explore the impact of integrating orthogonal data sources—such as short reads, CAGE peaks, and polyA motifs—on transcript validation and interpretation.

## 🔍 Rules exploration

1. From the set of rules on the file `data/filter_rules.json`, **are you able to write down the explanation of what they will check?**
    *example: Keep FSM isoforms that are up to 50 nt from the reference TSS OR are within a CAGE peak AND have no Retrotranscriptase switching AND there are no more than 60% of As downstream the TSS*

    <details><summary>Answer</summary>
    - ISM:
        - Keep isoforms with all junctions canonical AND no RT switching AND having no FSM in their associated transcripts AND having more than 50% of the total expression for their associated gene AND no intra-priming OR
        - Keep isoforms between 2000nt and 15000 nt of length AND being a 3' prime fragment, 5' fragment or internal fragment AND having no RT switching AND no intra-priming OR 
        - Keep isoforms with no RT switching AND no intra-priming AND with both their TSS and TTS supported by orthogonal data (CAGE peak and polyA motif).
    - NIC:
        - All their junctions have to be canonical OR being supported by more than 10 reads AND being within 50nts of the reference TSS AND TTS OR
        - Having at least 10 reads in all their junctions AND being part of a CAGE peak and have a polyA motif
    - rest:
        - Have no RT switching AND being coding AND having less than 60% of As after the TTS (intra-priming candidates) AND at least two exons AND all the junctions canonical OR at least 10 reads to support each junction.
    

    </details><br>

2. **Which rules do you expect that will make a difference between the complete and the basic run?**

<details><summary>Answer</summary>
    The rule that include the following parameters:

    - min_cov
    - within_CAGE_peak
    - polyA_motif_found
    - ratio_exp
Since they come from the orthogonal data only. For example, if a junction is not canonical but has at least 10 short reads that support it, it could be considered as valid, as the non-canonical junctions can also happen. 
</details><br>

3. **Can you put an example of an ISM that would be considered an artifact in the basic run but not in the complete run?**

<details><summary>Answer</summary>
    For this case, an ISM that has intron retention (the subcategory of `intron_retention` is considered an artifact here) would be eliminated in the basic run. However, this isoform would not be an artifact in the complete run if there are no FSMs for its associated gene (`FSM_class` B) and has 50% of the reads that mapped to the isoforms of the gene (`ratio_exp` > 0.5). As well, all of its junctions will have to be canonical. 
    </details><br>

## 🧠 Filter comparison

4. **How many isoforms are artifacts in the basic dataset?**

In [2]:
library(readr)
library(dplyr)

basic.df <- read_tsv("results/03_Filter_basic/human_chr8_RulesFilter_classification.txt",show_col_types = FALSE)
complete.df <- read_tsv("results/04_Filter_orthogonal/human_chr8_RulesFilter_classification.txt",show_col_types = FALSE)

basic.df %>% mutate(type="basic") %>%
    select(filter_result,type) %>%
    rbind(complete.df %>% mutate(type="complete") %>%
                          select(filter_result,type)) %>%
    table()

Warning message:
“package ‘dplyr’ was built under R version 4.5.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




             type
filter_result basic complete
     Artifact  1371      933
     Isoform   1081     1519

<details><summary>Answer</summary>1371 isoforms are considered artifacts in the basic dataset.</details><br>

5. **How many isoforms are artifacts in the complete dataset?**

In [2]:
# (See Question 4 - basic.df and complete.df are loaded and compared there)

<details><summary>Answer</summary>933 isoforms are considered artifacts in the complete dataset.</details><br>

6. **What is the distribution of the structural categories for the isoforms that passed the filter in the complete dataset but not in the basic dataset?**

In [5]:
pass_basic <- basic.df %>% filter(filter_result == "Isoform") %>%
    pull(isoform)

complete.df %>%
    filter(filter_result == "Isoform" & !isoform %in% pass_basic) %>%
    select(structural_category) %>%
    table()

structural_category
      full-splice_match incomplete-splice_match        novel_in_catalog 
                    245                      46                     147 

<details><summary>Answer</summary>

| Structural Category       | count |
|---------------------------|-------|
| full-splice_match         | 245   |
| incomplete-splice_match   | 46    |
| novel_in_catalog          | 147   |

</details><br>

7. **Why did those isoforms pass the filter in the complete dataset but not in the basic dataset?** *Pick one of each structural category.* *hint: Check the outputs produced by SQANTI filter*

In [10]:
complete.df %>% 
    filter(filter_result == "Isoform" & !isoform %in% pass_basic &
            structural_category == "full-splice_match") %>%
    select(isoform, structural_category, diff_to_TSS,within_CAGE_peak) %>% head(1)

complete.df %>% 
    filter(filter_result == "Isoform" & !isoform %in% pass_basic &
            structural_category == "incomplete-splice_match") %>%
    select(isoform, structural_category, length, diff_to_TSS,within_CAGE_peak,polyA_motif_found) %>% head(n=1)

complete.df %>%
   filter(filter_result == "Isoform" & !isoform %in% pass_basic &
            structural_category == "novel_in_catalog") %>%
    select(isoform, structural_category, length, subcategory,FSM_class,ratio_exp,within_CAGE_peak,polyA_motif_found) %>% head(n=1)

isoform,structural_category,diff_to_TSS,within_CAGE_peak
<chr>,<chr>,<dbl>,<lgl>
ENSG00000008513.17_4,full-splice_match,111,TRUE


isoform,structural_category,length,diff_to_TSS,within_CAGE_peak,polyA_motif_found
<chr>,<chr>,<dbl>,<dbl>,<lgl>,<lgl>
ENSG00000067167.8_3,incomplete-splice_match,1083,-1747,TRUE,TRUE


isoform,structural_category,length,subcategory,FSM_class,ratio_exp,within_CAGE_peak,polyA_motif_found
<chr>,<chr>,<dbl>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
ENSG00000040341.18_1,novel_in_catalog,2861,combination_of_known_junctions,C,NA,TRUE,TRUE


<details><summary>Answer</summary>

- **FSM**: The transcript `ENSG00000008513.17_4` is 111 nucleotides downstream of the reference TSS (exceeding the standard 50bp cutoff), but in the complete run it is supported by a CAGE peak, validating it.
- **ISM**: The 46 passing ISM isoforms, despite being incomplete transcripts, are validated by both a CAGE peak at the TSS and a polyA motif at the TTS, confirming they represent stable alternatives or actual partial products rather than degradation artifacts.

</details><br>

8. **Can you explain any biological reason why we might want to include one of those isoforms even thought they failed the basic filtering?**

<details><summary>Answer</summary>
For FSM transcripts like `ENSG00000008513.17_4`, which initiate transcription slightly further away from the annotated TSS than the arbitrary 50bp window allows, the existence of orthogonal validation (such as a CAGE peak) provides strong biological evidence that this is a true alternative transcription start site (TSS) rather than a degradation fragment.
</details><br>